In SQL 2025, you need Arc and must NOT have replication or CDC enabled!

https://learn.microsoft.com/en-us/fabric/mirroring/sql-server-limitations

https://learn.microsoft.com/en-us/fabric/mirroring/sql-server-tutorial?tabs=sql2025

In SQL 2016-2022 you NEED CDC.

In [ ]:
#r "nuget:Microsoft.DotNet.Interactive.SqlServer,*-*"

In [ ]:
#!connect mssql --kernel-name sqlserver --connection-string "Server=localhost\SQL22;TrustServerCertificate=True;Integrated Security=True"

In [ ]:
USE master
IF EXISTS (SELECT name FROM sys.databases WHERE name='MirroringDemo')BEGIN
    ALTER DATABASE MirroringDemo SET SINGLE_USER WITH ROLLBACK IMMEDIATE;
    DROP DATABASE MirroringDemo;
END
GO
CREATE DATABASE MirroringDemo
GO
USE MirroringDemo
GO
-- Customers
CREATE TABLE dbo.Customers
(
    CustomerID INT IDENTITY PRIMARY KEY,
    CustomerName NVARCHAR(100) NOT NULL,
    City NVARCHAR(50),
    Country NVARCHAR(50)
);
GO
-- Products
CREATE TABLE dbo.Products
(
    ProductID INT IDENTITY PRIMARY KEY,
    ProductName NVARCHAR(100) NOT NULL,
    Price DECIMAL(10,2) NOT NULL
);
GO 
-- Insert some sample data
INSERT INTO dbo.Customers (CustomerName, City, Country) VALUES
('Alice Smith', 'Berlin', 'Germany'),
('Bob Johnson', 'Seattle', 'USA'),
('Carla Ruiz', 'Madrid', 'Spain');

INSERT INTO dbo.Products (ProductName, Price) VALUES
('Laptop', 1200.00),
('Phone', 800.00),
('Headphones', 150.00),
('Keyboard', 50.00);

Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

...

(3 rows affected)

(4 rows affected)

(2 rows affected)

(2 rows affected)

(4 rows affected)

In [46]:
USE [master]
IF EXISTS (SELECT 1 FROM sys.server_principals  WHERE name = N'fabric_mirror')
BEGIN
    DROP LOGIN [fabric_mirror]
END
CREATE LOGIN [fabric_mirror] WITH PASSWORD = 'Mirror123!'
ALTER SERVER ROLE [sysadmin] ADD MEMBER [fabric_mirror]

Commands completed successfully.

In [47]:
USE MirroringDemo
CREATE USER [fabric_mirror] FOR LOGIN [fabric_mirror]
ALTER ROLE [db_owner] ADD MEMBER [fabric_mirror]

Commands completed successfully.

In [27]:
Start-Process https://learn.microsoft.com/en-us/data-integration/gateway/service-gateway-install

In [28]:
Invoke-WebRequest -Uri 'https://go.microsoft.com/fwlink/?linkid=2116849' -OutFile 'C:\Temp\OnPremGateway.exe'

Installation is pretty straightforward, requires Azure/Fabric user.

In [30]:
Start-Process C:\Temp\OnPremGateway.exe

In [32]:
Start-Process https://app.fabric.microsoft.com/groups/me/create?experience=fabric-developer

In [50]:
INSERT INTO dbo.Products 
SELECT 'Stickers', 1.00

(1 row affected)

Once CDC is enabled, you can remove fabric_login from the sysadmin server role and db_owner database role.

In [ ]:
USE MirroringDemo
ALTER ROLE [db_owner] DROP MEMBER [fabric_user]
GRANT CONNECT, SELECT to [fabric_user]

In [ ]:
USE [master]
ALTER SERVER ROLE [sysadmin] DROP MEMBER [fabric_login]
GRANT CONNECT, SELECT to [fabric_login]